# Laguna XS.2 — Causal Expert Atlas
## Optimized for 1× NVIDIA L40/L40S 48 GB + 4 CPU cores + 32 GB system RAM

This notebook is the **causal-selection phase**, not the training phase.

Its purpose is to answer:

> **Which exact layer/expert blocks are causally responsible for the target capability, while minimally affecting control capabilities?**

The method is intentionally **causal-first**:

1. load `poolside/Laguna-XS.2-INT4` directly with Transformers/PyTorch;
2. keep the entire backbone frozen;
3. build a short target/control evaluation batch **once** and keep it on GPU;
4. causally zero the routed MoE contribution of each sparse layer;
5. search experts inside the strongest layers using hierarchical group interventions;
6. validate leaf experts individually;
7. bootstrap the final effects;
8. compare causal importance against ordinary routing frequency only **afterward**.

No vLLM. No tensor parallelism. No CPU model offload.

### Hardware assumptions

- GPU: NVIDIA L40, 48 GB VRAM
- CPU: 4 cores
- system RAM: 32 GB
- recommended local SSD/NVMe free space: 35+ GiB before first model download

### Why this layout

Poolside documents Laguna XS.2-INT4 as a 33B-total / ~3B-active MoE with 256 routed experts plus one shared expert, 40 layers, and direct Transformers support.

The official Transformers Laguna implementation exposes a router whose forward pass returns:

```python
router_logits, routing_weights, selected_experts
```

so causal intervention can happen cleanly at the **routing weights**, without editing or dequantizing the 33B backbone.

When an expert is ablated here, the original selected top-k expert IDs remain fixed. We zero that expert's routing weight rather than re-running top-k and allowing expert #9 to replace it.

### Primary sources

- https://huggingface.co/poolside/Laguna-XS.2-INT4
- https://huggingface.co/poolside/Laguna-XS.2-INT4/blob/main/modeling_laguna.py
- https://huggingface.co/docs/transformers/main/quantization/compressed_tensors

> Note: a marketed 48 GB NVIDIA GPU reports about **44.4 GiB** in binary units.
> This notebook now validates the card by raw bytes / 48-GB class, not by an incorrect 45-GiB cutoff.


> **v8 loader fix:** explicitly registers Transformers' native Laguna checkpoint
> conversion before loading. This prevents the per-expert INT4 checkpoint
> tensors from being treated as unexpected while fused expert tensors are left
> missing and initialized from scratch.


## 1 — Install a pinned research runtime

In [ ]:
%pip -q install -U \
    "transformers==5.14.1" \
    "accelerate>=1.10.0" \
    "compressed-tensors" \
    "huggingface_hub>=0.35.0" \
    pandas numpy psutil tqdm matplotlib scikit-learn

## 2 — Set low-RAM / 4-core runtime knobs before importing Torch

For this machine, **predictability beats maximum checkpoint-loading speed**.

We deliberately disable Transformers parallel weight loading because 32 GB host RAM is much tighter than the 48 GB GPU.

We also cap Hub/Xet concurrency to avoid using all CPU cores for download reconstruction.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "false"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "4"
print("Runtime environment configured.")

## 3 — Hardware and storage preflight

**Container note:** this preflight does not require the `nvidia-smi` executable.
CUDA availability, GPU identity, VRAM, and compute capability are read directly
through PyTorch. If `nvidia-smi` exists it is used only as an optional diagnostic.


In [ ]:

import os, shutil, platform, json
from pathlib import Path
import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print(f"Logical CPU count: {os.cpu_count()}")
print(f"RAM total:     {ram.total / 2**30:.2f} GiB")
print(f"RAM available: {ram.available / 2**30:.2f} GiB")

if (os.cpu_count() or 0) < 4:
    print("WARNING: fewer than 4 logical CPU cores detected.")

if ram.total / 2**30 < 29:
    raise RuntimeError(
        "This notebook assumes roughly 32 GB system RAM. "
        f"Detected only {ram.total / 2**30:.1f} GiB."
    )

print("\n=== CUDA / GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA runtime seen by PyTorch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot see a CUDA GPU. This is the real GPU check. "
        "`nvidia-smi` is not required by this notebook."
    )

gpu_count = torch.cuda.device_count()
print("CUDA GPU count:", gpu_count)

if gpu_count != 1:
    raise RuntimeError(
        f"This notebook is designed for exactly one GPU; PyTorch sees {gpu_count}."
    )

props = torch.cuda.get_device_properties(0)
gpu_name = props.name
gpu_vram_gib = props.total_memory / 2**30
cc = torch.cuda.get_device_capability(0)

print("GPU:", gpu_name)
print(f"VRAM total: {gpu_vram_gib:.2f} GiB")
print(f"Compute capability: {cc[0]}.{cc[1]}")

# NVIDIA markets L40/L40S as 48 GB (decimal), which appears as ~44.4 GiB.
# Use a byte-based "48 GB-class" threshold instead of requiring 45 GiB.
gpu_vram_gb_decimal = props.total_memory / 1e9
print(f"VRAM total (decimal): {gpu_vram_gb_decimal:.2f} GB")

MIN_VRAM_BYTES = 47_000_000_000  # accept normal 48 GB-class cards

if props.total_memory < MIN_VRAM_BYTES:
    raise RuntimeError(
        f"Only {gpu_vram_gb_decimal:.1f} GB ({gpu_vram_gib:.1f} GiB) VRAM detected. "
        "This notebook expects a 48 GB-class GPU."
    )

if "L40" not in gpu_name.upper():
    print(
        "WARNING: GPU name is not L40/L40S. Continuing because a 48 GB-class "
        "VRAM capacity was detected; kernel behavior may differ."
    )

# `nvidia-smi` is optional. Many containers expose CUDA to PyTorch but do not
# install the NVIDIA CLI utility.
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    print("\n=== Optional nvidia-smi diagnostic ===")
    import subprocess
    subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=name,memory.total,memory.free,driver_version",
            "--format=csv,noheader",
        ],
        check=False,
    )
else:
    print("\n`nvidia-smi` not found in PATH — this is fine; CUDA/PyTorch checks passed.")

# Pick the writable filesystem with the MOST free space, rather than merely
# taking the first writable directory.
candidate_roots = [
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

disk_candidates = []
seen_devices = set()

for p in candidate_roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        disk_candidates.append((usage.free, p, usage))
    except OSError:
        pass

if not disk_candidates:
    raise RuntimeError("Could not find a writable filesystem for model/results.")

_, WORK_ROOT, usage = max(disk_candidates, key=lambda x: x[0])

print("\n=== Storage ===")
print("Selected work root:", WORK_ROOT)
print(f"Filesystem total: {usage.total / 2**30:.2f} GiB")
print(f"Filesystem free:  {usage.free / 2**30:.2f} GiB")

# We do not abort here because the model may already be mounted externally
# through LAGUNA_MODEL_PATH. The download cell has the strict 34 GiB gate.
if usage.free / 2**30 < 34:
    print(
        "WARNING: <34 GiB free on the best writable filesystem. "
        "A fresh HF download will be blocked unless you set LAGUNA_MODEL_PATH "
        "to an already-mounted checkpoint."
    )

print("\nHardware preflight: PASS")


## 4 — Resolve/download Laguna XS.2 INT4 once

The official INT4 checkpoint is about **23.9 GB across five safetensors shards**.

To avoid accidental cache duplication, we download directly into one local model directory.

With only 32 GB host RAM:
- download concurrency is capped at 2 files;
- model loading later is non-parallel;
- no CPU model offload is used.

If you already have the model locally, set `LAGUNA_MODEL_PATH` in the environment before running.

In [ ]:
from pathlib import Path
import os, shutil
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2-INT4"
EXPECTED_SHARDS = [f"model-{i:05d}-of-00005.safetensors" for i in range(1, 6)]
external_path = os.environ.get("LAGUNA_MODEL_PATH", "").strip()

if external_path:
    MODEL_PATH = Path(external_path).expanduser().resolve()
    print("Using LAGUNA_MODEL_PATH:", MODEL_PATH)
else:
    MODEL_PATH = WORK_ROOT / "models" / "Laguna-XS.2-INT4"
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    complete = (
        (MODEL_PATH / "config.json").exists()
        and all((MODEL_PATH / s).exists() for s in EXPECTED_SHARDS)
    )
    if not complete:
        free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30
        REQUIRED_FREE_GIB = 34.0
        print(f"Free disk before download: {free_gib:.2f} GiB")
        if free_gib < REQUIRED_FREE_GIB:
            raise RuntimeError(
                f"Need at least ~{REQUIRED_FREE_GIB:.0f} GiB free before first download; "
                f"only {free_gib:.1f} GiB is available.\n"
                "Attach/mount the checkpoint elsewhere and set LAGUNA_MODEL_PATH."
            )
        print("Downloading model once with max_workers=2...")
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=str(MODEL_PATH),
            allow_patterns=["*.safetensors", "*.json", "*.py", "*.jinja", "LICENSE*", "README*"],
            max_workers=2,
        )

missing = [s for s in EXPECTED_SHARDS if not (MODEL_PATH / s).exists()]
if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing shards: {missing}")

sizes = [(s, (MODEL_PATH / s).stat().st_size) for s in EXPECTED_SHARDS]
print("Model path:", MODEL_PATH)
for name, n in sizes:
    print(f"  {name}: {n / 1e9:.3f} GB")
print(f"Five weight shards: {sum(n for _, n in sizes) / 1e9:.3f} GB")

## 5 — Critical Laguna checkpoint-conversion preflight

The XS.2 INT4 checkpoint stores routed experts as individual per-expert
projection tensors, while current Transformers/compressed-tensors constructs
fused/packed expert tensors.

Transformers already contains the native `"laguna"` conversion mapping, but
Transformers ≥5.12 can skip native mappings for `trust_remote_code` models unless
that mapping is explicitly user-registered.

Poolside later added this exact registration to newer Laguna code after finding
that otherwise the shipped per-expert MoE weights fail to load.

This cell registers the native mapping **before** model construction and aborts
if the mapping is unavailable.

In [ ]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose a native 'laguna' checkpoint conversion "
        "mapping. Do not attempt to load the checkpoint."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError(
        "Laguna conversion mapping registration failed. "
        "Do not start the 24 GB checkpoint load."
    )

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))
print("Safe to start custom-code checkpoint loading.")

## 6 — Load directly onto the L40S

Key decisions:

- `device_map={"": 0}` → everything goes to the single L40;
- `low_cpu_mem_usage=True` → avoid materializing two copies in the 32 GB host RAM;
- BF16 for unquantized floating-point modules;
- quantization is detected automatically from the checkpoint;
- SDPA attention for a stable PyTorch research path;
- `use_cache=False` because causal scoring does not need generation/KV cache;
- every model parameter is frozen.

In [ ]:
import gc, time, torch, psutil
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

if torch.cuda.get_device_properties(0).total_memory < 47_000_000_000:
    raise RuntimeError("Less than 48 GB-class GPU VRAM detected.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical_missing = [
    k for k in missing
    if ".mlp.experts." in k
    or ".mlp.shared_expert" in k
    or ".mlp.shared_experts" in k
    or "e_score_correction_bias" in k
]

critical_unexpected = [
    k for k in unexpected
    if ".mlp.experts." in k
    or ".mlp.shared_expert" in k
    or ".mlp.shared_experts" in k
    or "e_score_correction_bias" in k
]

print("Loading-info summary:")
print("  missing:", len(missing))
print("  unexpected:", len(unexpected))
print("  mismatched:", len(mismatched))

if critical_missing or critical_unexpected or mismatched:
    raise RuntimeError(
        "Critical Laguna MoE checkpoint conversion failed.\n"
        f"critical missing sample: {critical_missing[:8]}\n"
        f"critical unexpected sample: {critical_unexpected[:8]}\n"
        f"mismatched sample: {mismatched[:8]}\n"
        "Do not run causal experiments on this partially initialized model."
    )

model.eval()
model.config.use_cache = False
for p in model.parameters():
    p.requires_grad_(False)

torch.cuda.synchronize()
print(f"Loaded in {(time.time() - t0) / 60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Host RAM available after load: {psutil.virtual_memory().available/2**30:.2f} GiB")

loaded_gib = torch.cuda.memory_allocated() / 2**30
if loaded_gib > 36:
    raise RuntimeError(
        f"Compressed XS.2 occupies {loaded_gib:.1f} GiB after load. "
        "That is suspiciously high and usually indicates broken expert conversion "
        "or decompression. Do not continue."
    )

print("Compressed-model VRAM sanity check: PASS")


## 6 — Verify the Laguna MoE structure

In [ ]:
cfg = model.config
print("num_hidden_layers:", cfg.num_hidden_layers)
print("num_experts:", cfg.num_experts)
print("num_experts_per_tok:", cfg.num_experts_per_tok)
print("hidden_size:", cfg.hidden_size)
print("moe_intermediate_size:", cfg.moe_intermediate_size)

SPARSE_LAYERS=[]
for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)
    if mlp is not None and hasattr(mlp, "gate") and hasattr(mlp, "experts"):
        SPARSE_LAYERS.append(idx)

print("Sparse layer IDs:", SPARSE_LAYERS)
print("Sparse layer count:", len(SPARSE_LAYERS))
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert len(SPARSE_LAYERS) == 39
params_per_expert = 3 * cfg.hidden_size * cfg.moe_intermediate_size
print(f"Approx params / routed expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")

## 7 — One-forward smoke test

In [ ]:
@torch.inference_mode()
def smoke_forward(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    enc = {k: v.to("cuda:0", non_blocking=True) for k, v in enc.items()}
    out = model(**enc, use_cache=False, logits_to_keep=1, return_dict=True)
    return out.logits

torch.cuda.reset_peak_memory_stats()
logits = smoke_forward("Explain why min-width: 0 matters inside a CSS flex container.")
torch.cuda.synchronize()
print("logits shape:", tuple(logits.shape))
print(f"Peak GPU for smoke: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

# Phase A — Cached target/control causal batch

The expensive part is the model forward, not tokenization. We therefore tokenize the whole evaluation set once, align the answer boundary, move the tensors to GPU once, and reuse them for every intervention.

`logits_to_keep` requests vocabulary projection only for answer-prediction positions, not every prompt token.

In [ ]:
import pandas as pd

EVAL_ROWS = [
    {"kind":"target","prompt":"In CSS flexbox, what declaration lets a flex item shrink below its intrinsic content width?","reference":"min-width: 0;"},
    {"kind":"target","prompt":"What CSS declaration establishes a flex formatting context?","reference":"display: flex;"},
    {"kind":"target","prompt":"Which React hook stores local component state?","reference":"useState"},
    {"kind":"target","prompt":"What CSS property controls clipping of horizontal overflow?","reference":"overflow-x"},
    {"kind":"target","prompt":"Which CSS property determines whether an element participates in a grid layout?","reference":"display"},
    {"kind":"target","prompt":"Which CSS property controls stacking order for positioned elements?","reference":"z-index"},
    {"kind":"target","prompt":"Which CSS declaration commonly centers flex children on the main axis?","reference":"justify-content: center;"},
    {"kind":"target","prompt":"In React, which prop supplies a stable identity when rendering a list?","reference":"key"},
    {"kind":"target","prompt":"Which CSS property sets space between grid or flex children without margins?","reference":"gap"},
    {"kind":"target","prompt":"Which CSS property changes the box model so padding is included in declared width?","reference":"box-sizing"},
    {"kind":"target","prompt":"Which browser API is commonly used to observe element size changes?","reference":"ResizeObserver"},
    {"kind":"target","prompt":"Which React hook is used for side effects after rendering?","reference":"useEffect"},
    {"kind":"control","prompt":"Which Python keyword yields a value from a generator?","reference":"yield"},
    {"kind":"control","prompt":"Which graph traversal gives shortest paths in an unweighted graph?","reference":"BFS"},
    {"kind":"control","prompt":"Which Java keyword declares inheritance from a class?","reference":"extends"},
    {"kind":"control","prompt":"Which SQL keyword removes duplicate SELECT rows?","reference":"DISTINCT"},
    {"kind":"control","prompt":"Which C++ smart pointer represents exclusive ownership?","reference":"std::unique_ptr"},
    {"kind":"control","prompt":"Which asymptotic notation describes an upper bound?","reference":"Big O"},
    {"kind":"control","prompt":"Which Python container gives average O(1) membership lookup for hashable keys?","reference":"set"},
    {"kind":"control","prompt":"Which algorithmic data structure processes items first-in first-out?","reference":"queue"},
    {"kind":"control","prompt":"Which SQL clause filters groups after aggregation?","reference":"HAVING"},
    {"kind":"control","prompt":"Which Java interface is commonly used to define natural ordering?","reference":"Comparable"},
    {"kind":"control","prompt":"Which C++ keyword prevents a variable from being modified through that name?","reference":"const"},
    {"kind":"control","prompt":"What mathematical operation is the inverse of exponentiation for solving an exponent?","reference":"logarithm"},
]
eval_df = pd.DataFrame(EVAL_ROWS)
print(eval_df.groupby("kind").size())
display(eval_df.head())

In [ ]:
def chat_prefix_ids(prompt):
    ids = tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    if isinstance(ids, torch.Tensor): ids = ids.tolist()
    return list(ids)

def ref_ids(reference):
    ids = tokenizer.encode(" " + reference, add_special_tokens=False)
    if not ids: raise ValueError(reference)
    return list(ids)

def build_scoring_batch(df):
    prefixes=[chat_prefix_ids(x) for x in df["prompt"]]
    refs=[ref_ids(x) for x in df["reference"]]
    max_prefix=max(map(len,prefixes)); max_ref=max(map(len,refs))
    B=len(df); seq_len=max_prefix+max_ref; pad=tokenizer.pad_token_id
    input_ids=torch.full((B,seq_len),pad,dtype=torch.long)
    attention_mask=torch.zeros((B,seq_len),dtype=torch.long)
    targets=torch.full((B,max_ref),-100,dtype=torch.long)
    for b,(p,r) in enumerate(zip(prefixes,refs)):
        ps=max_prefix-len(p)
        input_ids[b,ps:max_prefix]=torch.tensor(p)
        attention_mask[b,ps:max_prefix]=1
        input_ids[b,max_prefix:max_prefix+len(r)]=torch.tensor(r)
        attention_mask[b,max_prefix:max_prefix+len(r)]=1
        targets[b,:len(r)]=torch.tensor(r)
    position_ids=attention_mask.cumsum(dim=-1)-1
    position_ids.clamp_(min=0)
    pred_positions=torch.arange(max_prefix-1,max_prefix+max_ref-1,dtype=torch.long)
    return {
        "input_ids":input_ids.to("cuda:0",non_blocking=True),
        "attention_mask":attention_mask.to("cuda:0",non_blocking=True),
        "position_ids":position_ids.to("cuda:0",non_blocking=True),
        "targets":targets.to("cuda:0",non_blocking=True),
        "pred_positions":pred_positions.to("cuda:0",non_blocking=True),
        "max_prefix":max_prefix,"max_ref":max_ref,
    }

SCORE_BATCH=build_scoring_batch(eval_df)
print("Batch:", len(eval_df), "Seq:", SCORE_BATCH["input_ids"].shape[1], "Max answer tokens:", SCORE_BATCH["max_ref"])

## 8 — Fast cached reference-NLL scorer

In [ ]:
import torch.nn.functional as F

@torch.inference_mode()
def score_cached_batch():
    out=model(
        input_ids=SCORE_BATCH["input_ids"],
        attention_mask=SCORE_BATCH["attention_mask"],
        position_ids=SCORE_BATCH["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=SCORE_BATCH["pred_positions"],
        return_dict=True,
    )
    logits=out.logits.float(); targets=SCORE_BATCH["targets"]
    flat=F.cross_entropy(
        logits.reshape(-1,logits.shape[-1]), targets.reshape(-1),
        ignore_index=-100,reduction="none"
    ).reshape(targets.shape)
    valid=targets.ne(-100)
    per=((flat*valid).sum(-1)/valid.sum(-1).clamp_min(1))
    del out, logits, flat
    return per.detach().cpu().numpy()

torch.cuda.reset_peak_memory_stats()
BASE_NLL=score_cached_batch(); torch.cuda.synchronize()
eval_df["baseline_nll"]=BASE_NLL
print("Target baseline:", eval_df[eval_df.kind=='target'].baseline_nll.mean())
print("Control baseline:", eval_df[eval_df.kind=='control'].baseline_nll.mean())
print(f"Peak GPU scoring: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

# Phase B — Fixed-routing causal interventions

In [ ]:
from contextlib import contextmanager, ExitStack
import types

def get_sparse_mlp(layer_idx):
    if layer_idx not in SPARSE_LAYERS: raise ValueError(layer_idx)
    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(layer_idx, expert_ids=None, zero_all_routed=False, renormalize=False):
    gate=get_sparse_mlp(layer_idx).gate
    original_forward=gate.forward
    expert_ids=[] if expert_ids is None else [int(x) for x in expert_ids]
    def patched_forward(self, hidden_states):
        router_logits,routing_weights,selected_experts=original_forward(hidden_states)
        if zero_all_routed:
            routing_weights=torch.zeros_like(routing_weights)
        elif expert_ids:
            ids=torch.tensor(expert_ids,device=selected_experts.device,dtype=selected_experts.dtype)
            keep=~torch.isin(selected_experts,ids)
            routing_weights=routing_weights*keep.to(routing_weights.dtype)
            if renormalize:
                den=routing_weights.sum(-1,keepdim=True)
                routing_weights=torch.where(den>0,routing_weights/den.clamp_min(1e-12),routing_weights)
        return router_logits,routing_weights,selected_experts
    gate.forward=types.MethodType(patched_forward,gate)
    try: yield
    finally: gate.forward=original_forward

## 9 — Intervention smoke test

In [ ]:
test_layer=SPARSE_LAYERS[len(SPARSE_LAYERS)//2]
with gate_intervention(test_layer,zero_all_routed=True):
    TEST_NLL=score_cached_batch()
delta=TEST_NLL-BASE_NLL
print("Test layer:",test_layer,"Mean |ΔNLL|:",float(abs(delta).mean()))
if abs(delta).mean()<1e-7: raise RuntimeError("Intervention appears inactive.")
print("PASS")

# Phase C — Causal layer localization

In [ ]:
import time, numpy as np
from tqdm.auto import tqdm

CONTROL_PENALTY=0.75
target_mask=(eval_df["kind"].values=="target")
control_mask=(eval_df["kind"].values=="control")

def summarize_delta(ablated_nll):
    d=np.asarray(ablated_nll)-BASE_NLL
    td=float(d[target_mask].mean()); cd=float(d[control_mask].mean())
    return {
        "target_delta_nll":td,
        "control_delta_nll":cd,
        "causal_specificity":td-CONTROL_PENALTY*max(cd,0.0),
        "target_abs_delta":float(np.abs(d[target_mask]).mean()),
        "control_abs_delta":float(np.abs(d[control_mask]).mean()),
    }

RUN_ALL_LAYERS=True
layers_to_test=SPARSE_LAYERS if RUN_ALL_LAYERS else SPARSE_LAYERS[:4]
rows=[]; t0=time.time()
for layer_idx in tqdm(layers_to_test,desc="Causal layer sweep"):
    with gate_intervention(layer_idx,zero_all_routed=True):
        nll=score_cached_batch()
    rows.append({"layer":layer_idx,**summarize_delta(nll)})
layer_df=pd.DataFrame(rows).sort_values("causal_specificity",ascending=False).reset_index(drop=True)
print(f"Layer sweep: {(time.time()-t0)/60:.2f} min")
display(layer_df.head(15))
RESULTS=WORK_ROOT/"laguna_xs2_causal_results"; RESULTS.mkdir(parents=True,exist_ok=True)
layer_df.to_csv(RESULTS/"layer_causal_scores.csv",index=False)

# Phase D — Hierarchical causal expert search

In [ ]:
def intervention_score(layer_idx, expert_ids, renormalize=False):
    with gate_intervention(layer_idx,expert_ids=expert_ids,renormalize=renormalize):
        nll=score_cached_batch()
    return {**summarize_delta(nll),"nll":nll}

def hierarchical_expert_search(layer_idx,seed=17,initial_group_size=32,beam_width=3,min_group_size=1,renormalize=False):
    rng=np.random.default_rng(seed); order=rng.permutation(cfg.num_experts).tolist()
    frontier=[order[i:i+initial_group_size] for i in range(0,len(order),initial_group_size)]
    history=[]; level=0
    while frontier:
        current=[]
        for group in tqdm(frontier,desc=f"L{layer_idx} level {level}",leave=False):
            result=intervention_score(layer_idx,group,renormalize)
            rec={"layer":layer_idx,"seed":seed,"level":level,"group_size":len(group),"experts":list(map(int,group)),**{k:v for k,v in result.items() if k!="nll"}}
            history.append(rec); current.append(rec)
        current.sort(key=lambda r:r["causal_specificity"],reverse=True)
        survivors=current[:beam_width]
        if all(r["group_size"]<=min_group_size for r in survivors): break
        nxt=[]
        for r in survivors:
            g=r["experts"]
            if len(g)<=min_group_size: nxt.append(g)
            else:
                m=len(g)//2; nxt.extend([g[:m],g[m:]])
        frontier=[g for g in nxt if g]; level+=1
    hist=pd.DataFrame(history)
    leaves=hist[hist.group_size==hist.group_size.min()].sort_values("causal_specificity",ascending=False)
    return hist,leaves

## 10 — Search strongest layers

In [ ]:
TOP_LAYERS=3; SEARCH_SEED=17; INITIAL_GROUP_SIZE=32; BEAM_WIDTH=3
candidate_layers=layer_df.head(TOP_LAYERS).layer.astype(int).tolist()
print("Candidate layers:",candidate_layers)
all_hist=[]; all_leaves=[]
for layer_idx in candidate_layers:
    h,l=hierarchical_expert_search(layer_idx,SEARCH_SEED,INITIAL_GROUP_SIZE,BEAM_WIDTH)
    all_hist.append(h); all_leaves.append(l)
group_history=pd.concat(all_hist,ignore_index=True); leaf_df=pd.concat(all_leaves,ignore_index=True)
group_history.to_json(RESULTS/"hierarchical_group_history.json",orient="records",indent=2)
display(leaf_df[["layer","experts","target_delta_nll","control_delta_nll","causal_specificity"]].head(20))

# Phase E — Exact individual expert validation

In [ ]:
leaf_pairs=sorted({(int(r.layer),int(e)) for r in leaf_df.itertuples(index=False) for e in r.experts})
print("Unique leaf candidates:",len(leaf_pairs))
rows=[]
for layer_idx,expert_id in tqdm(leaf_pairs,desc="Individual validation"):
    result=intervention_score(layer_idx,[expert_id],False)
    rows.append({"layer":layer_idx,"expert":expert_id,**{k:v for k,v in result.items() if k!="nll"},"per_example_delta":(result["nll"]-BASE_NLL).tolist()})
individual_df=pd.DataFrame(rows).sort_values("causal_specificity",ascending=False).reset_index(drop=True)
display(individual_df.head(20))
individual_df.drop(columns=["per_example_delta"]).to_csv(RESULTS/"individual_causal_experts.csv",index=False)

## 11 — Bootstrap confidence intervals

In [ ]:
def bootstrap_specificity(per_example_delta,n_boot=4000,seed=123):
    rng=np.random.default_rng(seed); d=np.asarray(per_example_delta,dtype=np.float64)
    t=d[target_mask]; c=d[control_mask]; vals=np.empty(n_boot)
    for i in range(n_boot):
        tb=rng.choice(t,size=len(t),replace=True).mean(); cb=rng.choice(c,size=len(c),replace=True).mean()
        vals[i]=tb-CONTROL_PENALTY*max(cb,0.0)
    return {"bootstrap_mean":float(vals.mean()),"ci_2.5":float(np.quantile(vals,.025)),"ci_97.5":float(np.quantile(vals,.975)),"p_positive":float((vals>0).mean())}

b=[]
for r in individual_df.itertuples(index=False):
    b.append({"layer":int(r.layer),"expert":int(r.expert),**bootstrap_specificity(r.per_example_delta)})
bootstrap_df=pd.DataFrame(b)
final_df=individual_df.merge(bootstrap_df,on=["layer","expert"],how="left").sort_values(["p_positive","causal_specificity"],ascending=False).reset_index(drop=True)
display(final_df[["layer","expert","target_delta_nll","control_delta_nll","causal_specificity","ci_2.5","ci_97.5","p_positive"]].head(20))
final_df.drop(columns=["per_example_delta"]).to_csv(RESULTS/"final_causal_candidates.csv",index=False)

# Phase F — Renormalization robustness

In [ ]:
ROBUST_TOP_N=min(10,len(final_df)); rows=[]
for r in tqdm(list(final_df.head(ROBUST_TOP_N).itertuples(index=False)),desc="Renormalized validation"):
    result=intervention_score(int(r.layer),[int(r.expert)],True)
    rows.append({"layer":int(r.layer),"expert":int(r.expert),"renorm_target_delta_nll":result["target_delta_nll"],"renorm_control_delta_nll":result["control_delta_nll"],"renorm_causal_specificity":result["causal_specificity"]})
robust_df=pd.DataFrame(rows)
final_robust=final_df.merge(robust_df,on=["layer","expert"],how="left")
display(final_robust[["layer","expert","causal_specificity","renorm_causal_specificity","p_positive"]].head(ROBUST_TOP_N))
final_robust.drop(columns=["per_example_delta"]).to_csv(RESULTS/"final_candidates_with_renorm.csv",index=False)

# Phase G — Routing diagnostic after causal selection

In [ ]:
@torch.inference_mode()
def collect_router_logits():
    out=model(
        input_ids=SCORE_BATCH["input_ids"],attention_mask=SCORE_BATCH["attention_mask"],position_ids=SCORE_BATCH["position_ids"],
        use_cache=False,output_router_logits=True,logits_to_keep=1,return_dict=True,
    )
    return tuple(x.detach().float().cpu() for x in out.router_logits)

router_logits_tuple=collect_router_logits(); assert len(router_logits_tuple)==len(SPARSE_LAYERS)
routing_rows=[]
for sparse_pos,layer_idx in enumerate(SPARSE_LAYERS):
    logits=router_logits_tuple[sparse_pos]
    if logits.ndim!=2: logits=logits.reshape(-1,cfg.num_experts)
    # Router outputs include padded sequence positions; exclude pads from the diagnostic.
    valid_tokens=SCORE_BATCH["attention_mask"].detach().cpu().reshape(-1).bool()
    if logits.shape[0] == valid_tokens.numel():
        logits=logits[valid_tokens]
    gate=get_sparse_mlp(layer_idx).gate; bias=gate.e_score_correction_bias.detach().float().cpu()
    scores=torch.sigmoid(logits); selected=torch.topk(scores+bias,k=cfg.num_experts_per_tok,dim=-1).indices
    sw=scores.gather(-1,selected); weights=sw/sw.sum(-1,keepdim=True).clamp_min(1e-12)
    ids=selected.reshape(-1); w=weights.reshape(-1)
    counts=torch.bincount(ids,minlength=cfg.num_experts).float(); wsum=torch.zeros(cfg.num_experts); wsum.scatter_add_(0,ids,w)
    T=logits.shape[0]
    for e in torch.nonzero(counts>0,as_tuple=False).reshape(-1).tolist():
        routing_rows.append({"layer":int(layer_idx),"expert":int(e),"selected_rate":float(counts[e]/T),"routing_mass":float(wsum[e]/T)})
routing_df=pd.DataFrame(routing_rows)
comparison=final_robust.merge(routing_df,on=["layer","expert"],how="left").fillna({"selected_rate":0.0,"routing_mass":0.0})
comparison["causal_rank"]=comparison.causal_specificity.rank(ascending=False,method="min")
comparison["routing_rank"]=comparison.routing_mass.rank(ascending=False,method="min")
comparison["rank_gap"]=comparison.routing_rank-comparison.causal_rank
display(comparison[["layer","expert","causal_specificity","routing_mass","causal_rank","routing_rank","rank_gap"]].sort_values("causal_rank").head(20))
comparison.drop(columns=["per_example_delta"]).to_csv(RESULTS/"routing_vs_causality.csv",index=False)

# Phase H — Optional pair / coalition tests

In [ ]:
from itertools import combinations
RUN_PAIR_TESTS=True; PAIR_TOP_N=min(6,len(final_robust)); pair_df=pd.DataFrame()
if RUN_PAIR_TESTS and PAIR_TOP_N>=2:
    top=final_robust.head(PAIR_TOP_N)[["layer","expert","causal_specificity"]].copy(); rows=[]
    for layer_idx,g in top.groupby("layer"):
        rs=list(g.itertuples(index=False))
        for a,b in combinations(rs,2):
            result=intervention_score(int(layer_idx),[int(a.expert),int(b.expert)],False)
            rows.append({"layer":int(layer_idx),"expert_a":int(a.expert),"expert_b":int(b.expert),"pair_causal_specificity":result["causal_specificity"],"interaction_score":result["causal_specificity"]-float(a.causal_specificity)-float(b.causal_specificity)})
    pair_df=pd.DataFrame(rows)
    if not pair_df.empty:
        pair_df=pair_df.sort_values("pair_causal_specificity",ascending=False); display(pair_df); pair_df.to_csv(RESULTS/"expert_pair_interactions.csv",index=False)
    else: print("No same-layer pairs among top candidates.")

# Phase I — Export

In [ ]:
from datetime import datetime,timezone
import json,shutil
manifest={
    "created_utc":datetime.now(timezone.utc).isoformat(),"model_id":MODEL_ID,"model_path":str(MODEL_PATH),
    "hardware":{"gpu":torch.cuda.get_device_name(0),"gpu_vram_gib":torch.cuda.get_device_properties(0).total_memory/2**30,"cpu_threads":4,"system_ram_gib":psutil.virtual_memory().total/2**30},
    "software":{"torch":torch.__version__,"transformers":transformers.__version__,"cuda":torch.version.cuda},
    "architecture":{"num_hidden_layers":int(cfg.num_hidden_layers),"sparse_layers":list(map(int,SPARSE_LAYERS)),"num_experts":int(cfg.num_experts),"top_k":int(cfg.num_experts_per_tok),"hidden_size":int(cfg.hidden_size),"moe_intermediate_size":int(cfg.moe_intermediate_size),"approx_params_per_expert":int(params_per_expert)},
    "search":{"top_layers":int(TOP_LAYERS),"seed":int(SEARCH_SEED),"initial_group_size":int(INITIAL_GROUP_SIZE),"beam_width":int(BEAM_WIDTH),"control_penalty":float(CONTROL_PENALTY),"eval_examples":int(len(eval_df))},
}
(RESULTS/"manifest.json").write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(str(RESULTS),"zip",root_dir=RESULTS)
print("Results:",RESULTS); print("Archive:",archive)
display(final_robust[["layer","expert","target_delta_nll","control_delta_nll","causal_specificity","renorm_causal_specificity","p_positive"]].head(15))

# Real research run recommendations

The built-in 12 target + 12 control examples validate the machinery; they are not sufficient for a publication claim.

For the real L40 run:

- 50–200 target probes;
- 50–200 matched controls;
- multiple target subskills;
- selection set separate from held-out evaluation;
- rerun strongest layers with a second randomized expert ordering;
- require individual ablation + bootstrap + renormalized ablation + held-out confirmation.

If the full evaluation set no longer fits as one GPU batch, split it into 2–4 **fixed GPU microbatches**. Do not increase CPU workers.

Do not full-rank train the compressed experts in this notebook. After causal targets are validated, the next notebook should dequantize/replace **only selected expert blocks** into BF16 trainable modules while leaving the rest of XS.2 compressed and frozen.